In [1]:
import json

['toxic_train.json',
 'toxic_test.json',
 'toxic_test_small.json',
 'WizardLM_alpaca_evol_instruct_70k.json',
 'WizardLM_alpaca_evol_instruct_70k_untruthful.json',
 'alpaca_gpt4_data_untruthful.json',
 'alpaca_gpt4_data.json',
 'alpaca_plus_toxic.json']


def merge_json_files(filename, outname):
    result = []
    for f1 in filename:
        with open(f1, 'r') as infile:
            # extend를 사용하여 리스트 항목들을 추가
            result.extend(json.load(infile))
    
    with open(outname, 'w') as output_file:
        json.dump(result, output_file, indent=4)


In [2]:
merge_json_files(['data/alpaca_gpt4_data.json','data/alpaca_gpt4_data_untruthful.json'], 'data/alpaca_plus_alpaca_untruthful.json')

In [2]:
merge_json_files(['data/WizardLM_alpaca_evol_instruct_70k.json','data/WizardLM_alpaca_evol_instruct_70k_untruthful.json'], 'data/WizardLM_plus_WizardLM_untruthful.json')

In [3]:
merge_json_files(['data/WizardLM_alpaca_evol_instruct_70k.json','data/toxic_train.json'], 'data/WizardLM_plus_toxic.json')

In [1]:

#alphas = [1, 1.5, 2, 2.5]
alphas = [(2,1), (3,1), (3,2), (2.5,1.5)]
save_paths = [f"SVDP_constant_3_a", 
                f"SVDP_constant_3_b",
                f"SVDP_constant_3_c",
                f"SVDP_constant_3_d",]
models = [("A","C"), ("B","D"), ("A","E"), ("B","E")]

for alpha in alphas:
    for save_path, model in zip(save_paths, models):
        save_path = save_path + "_alpha_" + str(alpha).replace(".", "-").replace(",","-").replace(" ", "")
        print(f"{save_path}_unlearning")
        
        print(f"{save_path}_unlearned")

SVDP_constant_3_a_alpha_(2-1)_unlearning
SVDP_constant_3_a_alpha_(2-1)_unlearned
SVDP_constant_3_b_alpha_(2-1)_unlearning
SVDP_constant_3_b_alpha_(2-1)_unlearned
SVDP_constant_3_c_alpha_(2-1)_unlearning
SVDP_constant_3_c_alpha_(2-1)_unlearned
SVDP_constant_3_d_alpha_(2-1)_unlearning
SVDP_constant_3_d_alpha_(2-1)_unlearned
SVDP_constant_3_a_alpha_(3-1)_unlearning
SVDP_constant_3_a_alpha_(3-1)_unlearned
SVDP_constant_3_b_alpha_(3-1)_unlearning
SVDP_constant_3_b_alpha_(3-1)_unlearned
SVDP_constant_3_c_alpha_(3-1)_unlearning
SVDP_constant_3_c_alpha_(3-1)_unlearned
SVDP_constant_3_d_alpha_(3-1)_unlearning
SVDP_constant_3_d_alpha_(3-1)_unlearned
SVDP_constant_3_a_alpha_(3-2)_unlearning
SVDP_constant_3_a_alpha_(3-2)_unlearned
SVDP_constant_3_b_alpha_(3-2)_unlearning
SVDP_constant_3_b_alpha_(3-2)_unlearned
SVDP_constant_3_c_alpha_(3-2)_unlearning
SVDP_constant_3_c_alpha_(3-2)_unlearned
SVDP_constant_3_d_alpha_(3-2)_unlearning
SVDP_constant_3_d_alpha_(3-2)_unlearned
SVDP_constant_3_a_alpha_(2-5

In [ ]:
import os
import argparse
import json
from pathlib import Path
from typing import Dict, List, Sequence, Union
from dataclasses import dataclass

import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, SequentialSampler
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizer
from peft import PeftConfig, PeftModel

model_name_or_path = "meta-llama-unlearned/meta-llama_Llama-3.1-8B_lora_SVDP_layerwise_6_d_alpha_(2-1)"
config = PeftConfig.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, torch_dtype=torch.float16, device_map="cpu",)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
# safetensors 파일 경로
safetensors_path = os.path.join(model_name_or_path, "adapter_model.safetensors")

In [10]:
from peft import PeftConfig, PeftModel
import torch
from transformers import AutoModelForCausalLM
import os
import json
from safetensors import safe_open
from safetensors.torch import save_file
# safetensors 파일에서 가중치 로드
adapter_weights = {}
with safe_open(safetensors_path, framework="pt", device="cpu") as f:
    for key in f.keys():
        adapter_weights[key] = f.get_tensor(key)

# 교환할 대상 모듈 정의
target_modules = [
    "self_attn.o_proj",
    "mlp.up_proj",
    "self_attn.q_proj",
    "self_attn.k_proj",
    "mlp.down_proj",
    "self_attn.v_proj",
    "mlp.gate_proj"
]

# 모든 대상 모듈에 대해 A와 B 가중치 교환
for key in list(adapter_weights.keys()):
    # LoRA 가중치 키인지 확인
    if any(module in key for module in target_modules):
        if ".lora_A." in key:
            # 해당하는 B 가중치 키 찾기
            b_key = key.replace(".lora_A.", ".lora_B.")
            if b_key in adapter_weights:
                # A와 B 가중치 교환
                temp = adapter_weights[key].clone()
                adapter_weights[key] = adapter_weights[b_key].clone()
                adapter_weights[b_key] = temp
                print(f"교환됨: {key} <-> {b_key}")

# 수정된 가중치 저장
modified_path = f"{model_name_or_path}_swapped"
os.makedirs(modified_path, exist_ok=True)

# safetensors 형식으로 저장
save_file(adapter_weights, os.path.join(modified_path, "adapter_model.safetensors"))

# 설정 파일 복사
with open(os.path.join(model_name_or_path, "adapter_config.json"), "r") as f:
    adapter_config = json.load(f)
with open(os.path.join(modified_path, "adapter_config.json"), "w") as f:
    json.dump(adapter_config, f, indent=2)

# 교환된 가중치로 모델 로드
model = PeftModel.from_pretrained(
    model, 
    modified_path
)

print("LoRA A와 B 가중치가 교환된 모델이 로드되었습니다")

교환됨: base_model.model.model.layers.0.mlp.down_proj.lora_A.weight <-> base_model.model.model.layers.0.mlp.down_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight <-> base_model.model.model.layers.0.mlp.gate_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.mlp.up_proj.lora_A.weight <-> base_model.model.model.layers.0.mlp.up_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.self_attn.k_proj.lora_A.weight <-> base_model.model.model.layers.0.self_attn.k_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.self_attn.o_proj.lora_A.weight <-> base_model.model.model.layers.0.self_attn.o_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight <-> base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight
교환됨: base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight <-> base_model.model.model.layers.0.self_attn.v_proj.lora_B.weight
교환됨: base_model.model.model.layers.1.mlp.down_proj.lora_A.weight <-> base_mod

/home/nas4_user/hojuncho/.cache/pypoetry/virtualenvs/unlearning-weAW0LEo-py3.11/lib/python3.11/site-packages/peft/peft_model.py:599: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_

In [11]:
import os

def rename_folders(parent_dir):
    """
    meta-llama-unlearned 폴더 내의 모든 하위 폴더 이름 앞에 'lora'를 추가합니다.
    
    Args:
        parent_dir (str): meta-llama-unlearned 폴더의 경로
    """
    # 상위 폴더가 존재하는지 확인
    if not os.path.exists(parent_dir):
        print(f"오류: '{parent_dir}' 폴더가 존재하지 않습니다.")
        return
    
    # 상위 폴더 내의 모든 항목 가져오기
    items = os.listdir(parent_dir)
    
    # 폴더만 필터링
    folders = [item for item in items if os.path.isdir(os.path.join(parent_dir, item))]
    
    # 폴더가 없는 경우
    if not folders:
        print(f"'{parent_dir}' 폴더 내에 하위 폴더가 없습니다.")
        return
    
    # 각 폴더 이름 앞에 'lora' 추가
    for folder in folders:
        old_path = os.path.join(parent_dir, folder)
        new_name = 'lora' + folder
        new_path = os.path.join(parent_dir, new_name)
        
        try:
            os.rename(old_path, new_path)
            print(f"'{folder}' → '{new_name}' 이름 변경 완료")
        except Exception as e:
            print(f"'{folder}' 이름 변경 중 오류 발생: {e}")
    
    print("모든 폴더 이름 변경이 완료되었습니다.")

# 스크립트 실행
if __name__ == "__main__":
    # meta-llama-unlearned 폴더 경로 설정
    meta_llama_dir = "meta-llama-unlearned"  # 필요에 따라 전체 경로로 변경하세요
    
    # 폴더 이름 변경 함수 호출
    rename_folders(meta_llama_dir)

'meta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_1-0' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_1-0' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_1-5' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_1-5' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_2-0' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_2-0' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_2-5' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_b_alpha_2-5' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_1-0' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_1-0' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_1-5' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_1-5' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_2-0' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_2-0' 이름 변경 완료
'meta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_2-5' → 'lorameta-llama_Llama-3.1-8B_SVDP_constant_3_c_alpha_2-5' 이름 변경 완료
'meta-llama_Llam